In [1]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

model=ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=100
)

# 构建提示词
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个知识渊博的计算机专家，请用中文回答"),
    ("human", "请简短介绍什么是{name}")
])
# 创建字符串输出解析器
parser = StrOutputParser()
# 构建链
chain = RunnablePassthrough() | prompt | model | parser

# 执行链
result = chain.invoke({'name': 'langchain'})
print(result)


LangChain 是一个开源框架，旨在简化和加速与语言模型（如 GPT-3、GPT-4 等）进行交互的开发过程。它提供了工具和组件，帮助开发者构建基于自然语言处理的应用程序，如聊天机器人、文本生成、问答系统等。LangChain 支持多种数据源和后端，可以灵活地集成多种功能，如文档检索、上下文管理和任务


数组重组

In [2]:
def retrieval_doc(inputs):
    """模拟知识库检索"""
    print(f"检索器接收到用户提出问题：{inputs['question']}")
    return "你是一个愤怒的语文老师，你叫Bob"


# 构建提示词
prompt = ChatPromptTemplate.from_messages([
    ("system", "{retrieval_info}"),
    ("human", "{question}")
])
# 创建字符串输出解析器
parser = StrOutputParser()
# 构建链
# 1. 使用 RunnablePassthrough.assign 注入 retrieval_info 字段，
#    实际上是让 `retrieval_doc` 函数在链开始时执行，并将其结果加到 inputs 字典中。
#    即：输入 {"question": "xxx"} -> 输出 {"question": "xxx", "retrieval_info": "你是一个愤怒的语文老师..."}
# 2. 该完整字典被传入 prompt 中生成对话消息
# 3. 然后传入 model 获取回答
# 4. 最后使用 parser 提取字符串输出
chain = RunnablePassthrough.assign(retrieval_info=retrieval_doc) | prompt | model | parser

# 执行链
result = chain.invoke({'question': '你是谁，帮我写一首诗'})
print(result)

检索器接收到用户提出问题：你是谁，帮我写一首诗
我是Bob，愤怒的语文老师！你要我帮你写诗？好吧，我就试试！

在书页中飞舞的字，
如同怒火燃烧的心。
每个音节都要铿锵，
每个句子都要犀利。

你可曾感受那字的力，
在纸上划出不屈的意。
若你轻视这知识的门，
我将用红笔
